# Hands-on 3 — Similarity search: "find more like this"

**Chapter 4 · Working with embeddings**

The simplest thing you can do with embeddings is also one of the most useful: point at a place and ask for every other place that looks like it. No training, no labels, no thresholds on spectral bands. In this notebook you will:

1. build a **query** from one or several example pixels,
2. compute a similarity map and turn it into a **vector layer with areas**,
3. sharpen a query with **negative examples**,
4. scale up: search **all of Attica** (60 × 70 km) at 80 m resolution with a vector index (FAISS), the tool you would use for millions of locations.

Requires the cache from Hands-on 1. Expected run time about 3 minutes; ~45 MB downloaded in part 4.

In [ ]:
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio import features
from affine import Affine
from rasterio.crs import CRS
from shapely.geometry import shape
import folium

import eoemb

AOI = (23.90, 38.00, 24.00, 38.10)
cache = eoemb.DATA_DIR / "cache"

z = np.load(cache / "aef_attica_2024.npz")
transform = Affine.from_gdal(*z["transform"]); crs = CRS.from_string(str(z["crs"]))
emb = eoemb.dequantize(z["raw"]); H, W = emb.shape[1:]
extent = eoemb.lonlat_extent(transform, crs, (H, W))
with rasterio.open(eoemb.DATA_DIR / "s2_attica_20240713_rgb.tif") as src:
    s2_rgb = np.moveaxis(src.read(), 0, -1)
print("embeddings", emb.shape)

## 1. One example pixel is a fragile query

Let us look for **pine forest**. Start the way Hands-on 1 did, with one pixel on Penteli, and compare with a query built from five pixels.

Averaging embeddings is legitimate: the mean of unit vectors that point in similar directions is a vector pointing to the "centre" of that group. Re-normalise it and you have a **prototype** of the class.

In [ ]:
forest_pts = pd.DataFrame({"lon": [23.935, 23.928, 23.945, 23.922, 23.940],
                           "lat": [38.085, 38.090, 38.092, 38.080, 38.078]})
rows, cols = eoemb.lonlat_to_rowcol(forest_pts.lon, forest_pts.lat, transform, crs)
examples = eoemb.sample(emb, rows, cols)                       # (5, 64)

single = examples[0]
prototype = examples.mean(axis=0); prototype /= np.linalg.norm(prototype)

print("similarity of each example to the prototype:", np.round(examples @ prototype, 3))
print("similarity of each example to example #1:  ", np.round(examples @ single, 3))

sim_single = eoemb.cosine_similarity_map(emb, single)
sim_proto = eoemb.cosine_similarity_map(emb, prototype)

fig, ax = plt.subplots(1, 3, figsize=(19, 6.5))
ax[0].imshow(s2_rgb, extent=extent); ax[0].scatter(forest_pts.lon, forest_pts.lat, c="yellow", s=30); ax[0].set_title("Sentinel-2 with the five examples")
ax[1].imshow(sim_single, extent=extent, vmin=0.6, vmax=1, cmap="viridis"); ax[1].set_title("similarity to ONE example")
ax[2].imshow(sim_proto, extent=extent, vmin=0.6, vmax=1, cmap="viridis"); ax[2].set_title("similarity to the 5-pixel prototype")
plt.tight_layout()

The five examples are only 0.8 to 0.9 similar to each other: even inside one forest, pixels differ (density, slope, shadow, soil showing through). A single pixel therefore carries its own accidents into the query. The prototype averages them out and gives a cleaner map. **Always query with a handful of examples**, never with one.

## 2. From a similarity map to a map layer

A similarity map is a picture; a GIS wants polygons and areas. The steps are: threshold, clean up small specks, polygonise, compute area. The threshold is the one real decision, so look at what it does first.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5.5))
for a, thr in zip(ax, [0.80, 0.85, 0.90, 0.95]):
    mask = sim_proto > thr
    a.imshow(mask, extent=extent, cmap="Greens"); a.set_title(f"similarity > {thr}: {np.nansum(mask) / 100:.0f} ha")
plt.tight_layout()

There is no universally right threshold: 0.80 includes the scrubland south of the forest, 0.95 keeps only the densest stands. In practice you pick it by checking a few places you know, exactly as you would calibrate a vegetation-index threshold, but here the threshold is on *resemblance to your examples*, whatever combination of properties that involves.

We take 0.88 and remove patches smaller than 1 ha (100 pixels) to get rid of speckle.

In [ ]:
from scipy import ndimage

THR = 0.88
mask = sim_proto > THR
labels, n = ndimage.label(mask)
sizes = ndimage.sum(mask, labels, index=np.arange(1, n + 1))
keep = np.isin(labels, np.flatnonzero(sizes >= 100) + 1)
print(f"{n} patches above the threshold, {keep.sum() / 100:.0f} ha after removing patches < 1 ha")

polys = [(shape(geom), val) for geom, val in features.shapes(keep.astype(np.uint8), mask=keep, transform=transform)]
forest = gpd.GeoDataFrame({"class": ["pine forest"] * len(polys)}, geometry=[g for g, _ in polys], crs=crs)
forest["area_ha"] = forest.area / 10_000
forest = forest.sort_values("area_ha", ascending=False).reset_index(drop=True)
print(f"{len(forest)} polygons, largest {forest.area_ha.iloc[0]:.0f} ha, total {forest.area_ha.sum():.0f} ha")
forest.head()

In [ ]:
m = folium.Map(location=[38.05, 23.95], zoom_start=12, tiles="OpenStreetMap")
folium.GeoJson(forest.to_crs(4326), name=f"pine forest (sim > {THR})",
               style_function=lambda f: {"color": "darkgreen", "weight": 1, "fillOpacity": 0.45}).add_to(m)
for _, p in forest_pts.iterrows():
    folium.CircleMarker([p.lat, p.lon], radius=5, color="yellow", fill=True).add_to(m)
folium.LayerControl().add_to(m)
m

Save the layer if you want it in QGIS: `forest.to_file(cache / "pine_forest_2024.gpkg", driver="GPKG")`.

## 3. Negative examples

The 0.80 map above leaked into scrubland. Rather than pushing the threshold up (which also loses thin forest), tell the query what you do *not* want. A simple, effective recipe: subtract the mean of a few negative examples from the prototype, then re-normalise. This moves the query away from the direction that forest and scrub share.

In [ ]:
scrub_pts = pd.DataFrame({"lon": [23.905, 23.912, 23.900], "lat": [38.055, 38.048, 38.062]})
r2, c2 = eoemb.lonlat_to_rowcol(scrub_pts.lon, scrub_pts.lat, transform, crs)
negatives = eoemb.sample(emb, r2, c2)
neg_proto = negatives.mean(axis=0); neg_proto /= np.linalg.norm(neg_proto)
print("scrub prototype vs forest prototype similarity:", round(float(neg_proto @ prototype), 3))

ALPHA = 0.5
query = prototype - ALPHA * neg_proto
query /= np.linalg.norm(query)
sim_contrast = eoemb.cosine_similarity_map(emb, query)

fig, ax = plt.subplots(1, 3, figsize=(19, 6.5))
ax[0].imshow(s2_rgb, extent=extent)
ax[0].scatter(forest_pts.lon, forest_pts.lat, c="yellow", s=30, label="forest examples")
ax[0].scatter(scrub_pts.lon, scrub_pts.lat, c="red", s=30, label="scrub (negative)"); ax[0].legend(loc="lower left")
ax[1].imshow(sim_proto > 0.80, extent=extent, cmap="Greens"); ax[1].set_title("prototype only, > 0.80")
ax[2].imshow(sim_contrast > np.nanquantile(sim_contrast, 1 - np.nanmean(sim_proto > 0.80)), extent=extent, cmap="Greens")
ax[2].set_title("with negatives, same number of pixels")
plt.tight_layout()

Same number of selected pixels, but the selection has moved out of the scrub and into the forest. This is the embedding equivalent of "more like this, less like that", and it is how interactive tools such as Earth Genome's Earth Index refine a search from user clicks.

## 4. Search a whole region with a vector index

Similarity maps on one million pixels are cheap: a matrix–vector product. At regional or national scale you have hundreds of millions of vectors and many queries, and you want a **vector index**: a data structure that returns the *k* nearest neighbours of a query fast. FAISS (from Meta) is the standard open-source library; databases such as PostgreSQL with `pgvector` offer the same idea inside SQL.

To keep the download small we read AlphaEarth at 80 m instead of 10 m, using the COG overviews (`factor=8`). Each overview pixel is the re-normalised mean embedding of the 64 pixels under it, so it is still an embedding, just of a bigger area. Attica from Elefsina to the 24° E line: ~620 000 vectors.

In [ ]:
REGION = (23.30, 37.60, 24.00, 38.25)
t0 = time.time()
raw_r, tr_r, crs_r = eoemb.aef_read_bbox(REGION, 2024, factor=8)
emb_r = eoemb.dequantize(raw_r)
Hr, Wr = emb_r.shape[1:]
extent_r = eoemb.lonlat_extent(tr_r, crs_r, (Hr, Wr))
print(f"{emb_r.shape} at {tr_r.a:.0f} m, read in {time.time() - t0:.0f} s")

valid = ~np.isnan(emb_r[0])
X = np.ascontiguousarray(emb_r[:, valid].T, dtype=np.float32)   # (N, 64) unit vectors
pix_r, pix_c = np.where(valid)                                    # pixel position of each vector
print(f"{X.shape[0]:,} valid vectors")

plt.figure(figsize=(8, 9)); plt.imshow(eoemb.pca_rgb(emb_r), extent=extent_r); plt.title("Attica, AlphaEarth 2024 at 80 m, PCA(3)");

In [ ]:
import faiss

index = faiss.IndexFlatIP(64)   # inner product on unit vectors = cosine similarity; exact search
index.add(X)
print("index size:", index.ntotal)

def knn(lon, lat, k=30, exclude_km=3.0):
    """k most similar 80 m cells to the cell at (lon, lat), ignoring cells within exclude_km
    of the query (otherwise the answer is just the query's own neighbourhood)."""
    r, c = eoemb.lonlat_to_rowcol([lon], [lat], tr_r, crs_r)
    q = X[np.flatnonzero((pix_r == r[0]) & (pix_c == c[0]))]
    if len(q) == 0:
        raise ValueError("query falls on nodata")
    D, I = index.search(q, k * 20)
    d_px = np.hypot(pix_r[I[0]] - r[0], pix_c[I[0]] - c[0]) * tr_r.a / 1000
    far = d_px >= exclude_km
    hits = pd.DataFrame({"row": pix_r[I[0][far]], "col": pix_c[I[0][far]], "similarity": D[0][far]}).head(k)
    xs = tr_r.c + (hits.col + 0.5) * tr_r.a; ys = tr_r.f + (hits.row + 0.5) * tr_r.e
    lons, lats = rasterio.warp.transform(crs_r, "EPSG:4326", list(xs), list(ys))
    return hits.assign(lon=lons, lat=lats)

t0 = time.time()
ports = knn(23.62, 37.94)          # Piraeus container port
print(f"search took {time.time() - t0:.3f} s")
ports.head(10)

In [ ]:
def show_hits(hits, title, query_lonlat):
    fig, ax = plt.subplots(figsize=(9, 10))
    ax.imshow(eoemb.pca_rgb(emb_r), extent=extent_r, alpha=0.6)
    ax.scatter(hits.lon, hits.lat, s=60, facecolors="none", edgecolors="red", linewidths=1.5, label="nearest neighbours")
    ax.plot(*query_lonlat, "y*", ms=18, mec="k", label="query"); ax.legend(loc="lower left"); ax.set_title(title)

show_hits(ports, "30 cells most similar to Piraeus port, more than 3 km away", (23.62, 37.94))

The neighbours cluster around the other harbours and industrial waterfronts of the Saronic Gulf: Perama, Salamina, Elefsina, the Lavrio side is outside the region. A "find all ports" map from one click and 0.01 s of search.

Try the airport: does the model find other large paved areas?

In [ ]:
airport = knn(23.944, 37.936)
show_hits(airport, "cells most similar to Athens airport runway", (23.944, 37.936))
airport.head()

Interactive version: same hits on a web map, click a marker for the similarity value.

In [ ]:
m = folium.Map(location=[37.95, 23.65], zoom_start=10, tiles="OpenStreetMap")
for _, h in ports.iterrows():
    folium.CircleMarker([h.lat, h.lon], radius=6, color="red", fill=True, popup=f"sim {h.similarity:.3f}").add_to(m)
folium.Marker([37.94, 23.62], popup="query: Piraeus port", icon=folium.Icon(color="orange", icon="star")).add_to(m)
m

### Exact or approximate?

`IndexFlatIP` compares the query with every vector: exact, and fast enough up to a few million vectors on a laptop (about 1 ms per 100 000 vectors). Beyond that, FAISS offers approximate indexes (`IndexIVFFlat`, `IndexHNSWFlat`) that trade a little recall for orders of magnitude in speed and memory. Major TOM ships such an index for its 170 million embeddings. The workflow lesson in Chapter 6 comes back to this.

---
## Exercises

**E1.** Build a prototype for **built-up** land from five pixels in Nea Makri and produce a polygon layer of settlements in the study area. Compare its total area with the ESA WorldCover "built-up" class, which the course ships on the same grid (`data/worldcover_2021_attica.tif`, class value 50).

**E2.** In part 4, query the **Marathon reservoir** (23.90, 38.16) with `exclude_km=1`. What do the neighbours have in common? Now query a beach (Schinias is outside the region; try Varkiza at (23.80, 37.82)).

**E3.** Time the search for 1 000 random queries with `IndexFlatIP`. Then build `faiss.IndexHNSWFlat(64, 32)` (with `index.metric_type = faiss.METRIC_INNER_PRODUCT` before adding vectors), and compare speed and how many of the top-10 neighbours agree with the exact index.